In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import xgboost as xgb
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

## Przekształcenie danych

In [8]:
test = pd.read_csv("../data/processed/real_estate_ads_last6m_test.csv")
train = pd.read_csv("../data/processed/real_estate_ads_last6m_train.csv")

In [9]:
test.shape # (737, 15)
train.shape # (2950, 15)

(2950, 15)

In [10]:
# braki danych
test['district_lon'] = test['district_lon'].fillna(train['district_lon'].median())
test['district_lat'] = test['district_lat'].fillna(train['district_lat'].median())
test['title'] = test['title'].fillna("")

train = train.dropna() # 2813

In [11]:
test = test.drop(columns = "id")
train = train.drop(columns = "id")

# usuwamy duplikaty ze zbioru treningowego
identyfikatory = ['map_lat', 'map_lon', 'm', 'no_rooms', 'district_lon', 'district_lat']

train['created_at'] = pd.to_datetime(train['created_at'])
train = train.sort_values(by='created_at', ascending=False)
train = train.drop_duplicates(subset=identyfikatory, keep='first')
# w train mieliśmy 2859 duplikatów

# tworzymy zmienną długość opisów
dlugosc_opisow = test['description'].fillna('').apply(len)
test['desc_length'] = dlugosc_opisow

dlugosc_opisow = train['description'].fillna('').apply(len)
train['desc_length'] = dlugosc_opisow

# zmienna no_rooms
test['no_rooms'] = test['no_rooms'].replace('more', '8')
test['no_rooms'] = test['no_rooms'].astype('int64')

train['no_rooms'] = train['no_rooms'].replace('more', '8')
train['no_rooms'] = train['no_rooms'].astype('int64')

# zmienna market
test['market'] = test['market'] == 'secondary'
train['market'] = train['market'] == 'secondary'

In [12]:
# Wykrywamy outliery w zbiorze treningowym
def get_outlier_mask(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return (df[column] < lower_bound) | (df[column] > upper_bound)

mask_rooms = get_outlier_mask(train, 'no_rooms')
mask_m = get_outlier_mask(train, 'm')
mask_price_m = get_outlier_mask(train, 'price_per_m')
mask_price = get_outlier_mask(train, 'price')
total_outliers_mask = mask_rooms | mask_m | mask_price_m | mask_price

train = train[~total_outliers_mask] # usuwamy outliery
print(f"Liczba usuniętych wierszy łącznie {total_outliers_mask.sum()}") # 258

Liczba usuniętych wierszy łącznie 258


In [13]:
def expand_params_to_separate_df(df, params_col='params'):
    expanded_list = []
    for row in df[params_col]:
        row_dict = {}
        if pd.notna(row):
            pairs = row.split('<br>') # rozdzielamy parametry po znaczniku <br>
            for pair in pairs:
                if '<=>' in pair:
                    key, value = pair.split('<=>', 1)
                    if key != value:
                        row_dict[key] = value
        expanded_list.append(row_dict)
    
    params_df = pd.DataFrame(expanded_list)
    original_cols = df.columns.tolist()
    new_cols = [col for col in params_df.columns if col not in original_cols]
    return params_df[new_cols]

test_params = expand_params_to_separate_df(test)
test_params = test_params.drop(columns=['rooms_num'])

train_params = expand_params_to_separate_df(train)
train_params = train_params.drop(columns=['rooms_num'])


In [14]:
wartosci_do_usuniecia = ['', ' ', 'NaN', 'nan', 'None', 'brak informacji']
test_params = test_params.replace(wartosci_do_usuniecia, np.nan)
train_params = train_params.replace(wartosci_do_usuniecia, np.nan)

In [15]:
def clean_floor_no(df_p):
    mapping = {'ground_floor': '0', 'cellar': '-1'}
    df_p['floor_no'] = df_p['floor_no'].replace(mapping)
    df_p['floor_no'] = df_p['floor_no'].astype(str).str.extract(r'(-?\d+)').astype(float)
    return df_p

def clean_building(df_p):
    mapping = {'block': '0', 'apartment': '1', 'ribbon': '2', 'tenement': '3', 'house': '4'}
    df_p['building_type'] = df_p['building_type'].map(mapping)
    return df_p

In [16]:
test_params['remote_services'] = (test_params['remote_services'].fillna(0).astype(int))
test_params = clean_floor_no(test_params)
test_params['floor_no'].isna().sum() # 21
test_params = clean_building(test_params)
test_params['building_type'].isna().sum() # mamy 115 braków
test_params['building_floors_num'].isna().sum() # mamy 71 braków

train_params['remote_services'] = (train_params['remote_services'].fillna(0).astype(int))
train_params = clean_floor_no(train_params)
train_params['floor_no'].isna().sum()
train_params = clean_building(train_params)
train_params['building_type'].isna().sum()
train_params['building_floors_num'].isna().sum()

# w zbiorze testowym i treningowym wszystkie braki uzupełniamy za pomocą najczęstszej wartości ze zbioru train
mode_floor = train_params['floor_no'].mode()[0]
mode_building_type = train_params['building_type'].mode()[0]
mode_building_floors = train_params['building_floors_num'].mode()[0]

train_params['floor_no'] = train_params['floor_no'].fillna(mode_floor)
test_params['floor_no'] = test_params['floor_no'].fillna(mode_floor)

train_params['building_type'] = train_params['building_type'].fillna(mode_building_type)
test_params['building_type'] = test_params['building_type'].fillna(mode_building_type)

train_params['building_floors_num'] = train_params['building_floors_num'].fillna(mode_building_floors)
test_params['building_floors_num'] = test_params['building_floors_num'].fillna(mode_building_floors)


In [17]:
def process_extras(df_p, col_name='extras_types'):
    extras_series = df_p[col_name].fillna('') # zamieniamy Nany na puste
    extras_dummies = extras_series.str.get_dummies(sep='<->')
    extras_dummies = extras_dummies.add_prefix('extras_')
    return extras_dummies

test_extras_encoded = process_extras(test_params)
train_extras_encoded = process_extras(train_params)

test_extras_encoded = test_extras_encoded.drop(columns = 'extras_0')
train_extras_encoded = train_extras_encoded.drop(columns = 'extras_0')

In [18]:
params_test = test_params[['remote_services', 'floor_no', 'building_type', 'building_floors_num']]
params_test = pd.concat([params_test, test_extras_encoded], axis=1)

params_train = train_params[['remote_services', 'floor_no', 'building_type', 'building_floors_num']]
params_train = pd.concat([params_train, train_extras_encoded], axis=1)

In [19]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

city_center_lat = 52.2420
city_center_lon = 16.5525
station_lat = 52.4010
station_lon = 16.9080

test['distance_to_center'] = haversine(test['map_lat'], test['map_lon'], city_center_lat, city_center_lon)
test['distance_to_station'] = haversine(test['map_lat'], test['map_lon'],station_lat,station_lon)
train['distance_to_center'] = haversine(train['map_lat'], train['map_lon'], city_center_lat, city_center_lon)
train['distance_to_station'] = haversine(train['map_lat'], train['map_lon'],station_lat,station_lon)

In [20]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'\s+', ' ', text) # usuwamy tagi HTML
    text = re.sub(r'[^a-zA-ZąęćłńóśźżĄĘĆŁŃÓŚŹŻ\s]', ' ', text) # usuwamy znaki interpunkcyjne 
    text = text.lower().strip() # małe litery
    return text

test['clean_description'] = test['description'].apply(clean_text)
test['clean_title'] = test['title'].apply(clean_text)
train['clean_description'] = train['description'].apply(clean_text)
train['clean_title'] = train['title'].apply(clean_text)

In [21]:
# dane do modelowania
dane_test = test.drop(columns = ['params', 'price', 'title', 'description'])
dane_test = dane_test.reset_index(drop=True)
dane_test = pd.concat([dane_test, params_test], axis=1)

dane_train = train.drop(columns = ['params', 'price', 'title', 'description'])
dane_train = dane_train.reset_index(drop=True)
dane_train = pd.concat([dane_train, params_train], axis=1)

In [22]:
# zmieniamy typ danych
dane_test["building_type"] = dane_test["building_type"].astype(int)
dane_test["building_floors_num"] = dane_test["building_floors_num"].astype(int)
dane_train["building_type"] = dane_train["building_type"].astype(int)
dane_train["building_floors_num"] = dane_train["building_floors_num"].astype(int)

In [23]:
features_to_drop = ['created_at', 'created_at_first', "price_per_m"]
X_train = dane_train.drop(columns=features_to_drop)
y_train = dane_train["price_per_m"]

X_test = dane_test.drop(columns=features_to_drop)
y_test = dane_test["price_per_m"]

In [24]:
# zmienne do baseline
data_base = ['district_lon', 'district_lat', 'no_rooms', 'm', 'map_lon', 'map_lat']

X_train_base = X_train[data_base]
X_test_base = X_test[data_base]

## Tfidf

In [25]:
# dla description
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=5, max_df=0.9)
X_train_tfidf = tfidf.fit_transform(X_train['clean_description'])
X_test_tfidf = tfidf.transform(X_test['clean_description'])

n_components = 60
svd = TruncatedSVD(n_components=n_components, random_state=443712)
X_train_lsa = svd.fit_transform(X_train_tfidf)
X_test_lsa = svd.transform(X_test_tfidf)

lsa_columns = [f'tfidf_lsa_{i}' for i in range(n_components)]

X_train_lsa_df = pd.DataFrame(X_train_lsa, columns=lsa_columns, index=X_train.index)
X_test_lsa_df = pd.DataFrame(X_test_lsa, columns=lsa_columns, index=X_test.index)

X_train_full_tf = pd.concat([X_train, X_train_lsa_df], axis=1)
X_test_full_tf = pd.concat([X_test, X_test_lsa_df], axis=1)

In [26]:
# dla title
tfidf2 = TfidfVectorizer(max_features=500, ngram_range=(1, 2), min_df=5, max_df=0.9)
X_train_tfidf2 = tfidf2.fit_transform(X_train['clean_title'])
X_test_tfidf2 = tfidf2.transform(X_test['clean_title'])

n_components = 30
svd2 = TruncatedSVD(n_components=n_components, random_state=443712)
X_train_lsa2 = svd2.fit_transform(X_train_tfidf2)
X_test_lsa2 = svd2.transform(X_test_tfidf2)

lsa_columns_title = [f'tfidf_title_lsa_{i}' for i in range(n_components)]

X_train_lsa_df2 = pd.DataFrame(X_train_lsa2, columns=lsa_columns_title, index=X_train.index)
X_test_lsa_df2 = pd.DataFrame(X_test_lsa2, columns=lsa_columns_title, index=X_test.index)

X_train_full_tf = pd.concat([X_train_full_tf, X_train_lsa_df2], axis=1)
X_test_full_tf = pd.concat([X_test_full_tf, X_test_lsa_df2], axis=1)

In [27]:
# usuwamy zmienne ['clean_description', 'extras_', 'clean_title'] ze zbioru treningowego i testowego
col = ['clean_description', 'extras_', 'clean_title']
X_train_full_tf = X_train_full_tf.drop(columns = col, errors = 'ignore')
X_test_full_tf = X_test_full_tf.drop(columns = col, errors = 'ignore')

## Open AI

In [118]:
fea_test = pd.read_csv(("features_test.csv"))
fea_train = pd.read_csv(("features_train.csv"))

X_test_open = pd.concat([X_test, fea_test], axis=1)
X_train_open = pd.concat([X_train, fea_train], axis=1)

col = ['clean_description', 'extras_', 'clean_title']
X_test_open = X_test_open.drop(columns = col, errors = 'ignore')
X_train_open = X_train_open.drop(columns = col, errors = 'ignore')

## Model

In [119]:
def evaluate_model(y_true, y_pred, model_name, dataset_name):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    print(f"Wyniki na zbiorze testowym {model_name} ({dataset_name})")
    print(f"R^2:  {r2:.4f}")
    print(f"MAE:  {mae:.2f} PLN")
    print(f"RMSE: {rmse:.2f} PLN")
    print(f"MAPE: {mape:.2%}")

    return {"Model": model_name, "": dataset_name, "R2": r2, "MAE": mae, "RMSE": rmse, "MAPE": mape}

all_results = []

In [120]:
xgb_model_base = xgb.XGBRegressor(learning_rate=0.1, max_depth=5, n_estimators=500, random_state = 441273, n_jobs = -1)

xgb_model_base.fit(X_train_base, y_train)
y_pred_base = xgb_model_base.predict(X_test_base)

results_baseline = evaluate_model(y_test, y_pred_base, "XGBoost", "Baseline")
all_results.append(results_baseline)

Wyniki na zbiorze testowym XGBoost (Baseline)
R^2:  0.2754
MAE:  461.53 PLN
RMSE: 752.36 PLN
MAPE: 5.70%


In [121]:
xgb_model = xgb.XGBRegressor(learning_rate=0.1, max_depth=5, n_estimators=500, random_state = 441273, n_jobs = -1)

xgb_model.fit(X_train_full_tf, y_train)
y_pred_tfidf = xgb_model.predict(X_test_full_tf)

results_tf = evaluate_model(y_test, y_pred_tfidf, "XGBoost", "TF-IDF")
all_results.append(results_tf)

Wyniki na zbiorze testowym XGBoost (TF-IDF)
R^2:  0.4837
MAE:  388.89 PLN
RMSE: 635.12 PLN
MAPE: 4.76%


In [122]:
xgb_model_open = xgb.XGBRegressor(learning_rate=0.1, max_depth=5, n_estimators=500, random_state = 441273, n_jobs = -1)

xgb_model_open.fit(X_train_open.values, y_train)
y_pred_open = xgb_model_open.predict(X_test_open.values)

results_open = evaluate_model(y_test, y_pred_open, "XGBoost", "OpenAI")
all_results.append(results_open)

Wyniki na zbiorze testowym XGBoost (OpenAI)
R^2:  0.4202
MAE:  421.58 PLN
RMSE: 673.02 PLN
MAPE: 5.20%


In [125]:
comparison = pd.DataFrame(all_results)
display(comparison)
comparison.to_csv('comparison.csv', index=False)

,Model,,R2,MAE,RMSE,MAPE
0,XGBoost,Baseline,0.275437,461.529449,752.364209,0.056956
1,XGBoost,TF-IDF,0.483667,388.892734,635.118459,0.047551
2,XGBoost,OpenAI,0.420210,421.581269,673.016165,0.052015
